<a href="https://colab.research.google.com/github/KSaubhagya/Prediction-Module/blob/main/Risk_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer


2.Load Data

In [20]:
from google.colab import files
uploaded = files.upload()


Saving dataset final - Prediction Module subset.csv to dataset final - Prediction Module subset (1).csv


In [21]:
df = pd.read_csv(list(uploaded.keys())[0])

In [22]:
# Standardize column names
df.columns = df.columns.str.strip()
print(df.columns.tolist())
df.head()

['Name', 'student_id', 'gender', 'program', 'gpa', 'previous_results', 'module_code', 'course_information', 'module_credits', 'ca_weight', 'final_exam_weight', 'ca_marks', 'final_exam_marks', 'total_marks', 'final_grade', 'lecture_attendance', 'attendance_percentage', 'assignment_marks', 'exam_results', 'late_arrivals', 'quiz_scores', 'submission_status', 'login_frequency', 'time_spent_lms', 'resources_accessed', 'forum_chat_activity', 'course_start_date', 'education_level', 'click_count', 'assessment_date', 'assessment_type', 'assessment_score', 'assessment_max_score', 'Learning_Style', 'week_number', 'week_start_date', 'week_end_date', 'assignment_due_date', 'assignment_submission_date', 'sessions_attended', 'total_sessions_held', 'click_timestamp', 'module_id', 'pass_fail_status', 'assignment_max_marks']


,Name,student_id,gender,program,gpa,previous_results,module_code,course_information,module_credits,ca_weight,...,week_start_date,week_end_date,assignment_due_date,assignment_submission_date,sessions_attended,total_sessions_held,click_timestamp,module_id,pass_fail_status,assignment_max_marks
0,Nadeesha Baskaran,ST725,Female,IT,3.11,Good,IN2101,Object Oriented Programming,3.0,40,...,2025-06-30,2025-07-06,2025-07-08,2025-07-07,24,25,2025-05-05 20:40:00,MOD-IN2101,Pass,40
1,Amila Piyarathna,ST383,Male,IT,3.02,Good,IN2901,Software Development Project,4.0,100,...,2025-07-06,2025-07-12,2025-07-12,2025-07-15,15,27,2025-05-25 13:19:00,MOD-IN2901,Pass,100
2,Amila Piyarathna,ST383,Male,IT,3.02,Good,CM2131,Essentials of Mathematical Methods,2.5,30,...,2025-04-27,2025-05-03,2025-05-04,2025-05-08,18,26,2025-05-25 14:00:00,MOD-CM2131,Pass,30
3,Amila Piyarathna,ST383,Male,IT,3.02,Good,IN2101,Object Oriented Programming,3.0,40,...,2025-04-13,2025-04-19,2025-04-20,2025-04-24,12,25,2025-05-25 12:57:00,MOD-IN2101,Pass,40
4,Amila Piyarathna,ST383,Male,IT,3.02,Good,IN2321,Computer Architecture,2.5,30,...,2025-05-11,2025-05-17,2025-05-18,2025-05-22,18,30,2025-05-25 13:11:00,MOD-IN2321,Pass,30


3. Initial Cleaning

In [23]:
# --- Remove duplicates ---
print("Duplicates found:", df.duplicated().sum())
df = df.drop_duplicates()


Duplicates found: 0


In [24]:
# --- Check missing values ---
print(df.isnull().sum())

Name                          0
student_id                    0
gender                        0
program                       0
gpa                           0
previous_results              0
module_code                   0
course_information            0
module_credits                0
ca_weight                     0
final_exam_weight             0
ca_marks                      0
final_exam_marks              0
total_marks                   0
final_grade                   0
lecture_attendance            0
attendance_percentage         0
assignment_marks              0
exam_results                  0
late_arrivals                 0
quiz_scores                   0
submission_status             0
login_frequency               0
time_spent_lms                0
resources_accessed            0
forum_chat_activity           0
course_start_date             0
education_level               0
click_count                   0
assessment_date               0
assessment_type               0
assessme

4.Handle Missing Values

In [25]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

In [26]:
print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

Numerical columns: ['gpa', 'module_credits', 'ca_weight', 'final_exam_weight', 'ca_marks', 'final_exam_marks', 'total_marks', 'lecture_attendance', 'attendance_percentage', 'assignment_marks', 'exam_results', 'late_arrivals', 'quiz_scores', 'login_frequency', 'time_spent_lms', 'resources_accessed', 'forum_chat_activity', 'click_count', 'assessment_score', 'assessment_max_score', 'sessions_attended', 'total_sessions_held', 'assignment_max_marks']
Categorical columns: ['Name', 'student_id', 'gender', 'program', 'previous_results', 'module_code', 'course_information', 'final_grade', 'submission_status', 'course_start_date', 'education_level', 'assessment_date', 'assessment_type', 'Learning_Style', 'week_number', 'week_start_date', 'week_end_date', 'assignment_due_date', 'assignment_submission_date', 'click_timestamp', 'module_id', 'pass_fail_status']


In [27]:
# Impute numerical missing values with median
num_imputer = SimpleImputer(strategy='median')
df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

In [28]:
# Impute categorical missing values with most frequent
cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

5. CATEGORICAL ENCODING (One-Hot)

In [29]:
# Identify date columns to exclude from one-hot encoding
date_like_cols = ['course_start_date', 'assessment_date']
ohe_cols = [c for c in categorical_cols if c not in date_like_cols]

In [30]:
print("Columns to one-hot encode:", ohe_cols)

Columns to one-hot encode: ['Name', 'student_id', 'gender', 'program', 'previous_results', 'module_code', 'course_information', 'final_grade', 'submission_status', 'education_level', 'assessment_type', 'Learning_Style', 'week_number', 'week_start_date', 'week_end_date', 'assignment_due_date', 'assignment_submission_date', 'click_timestamp', 'module_id', 'pass_fail_status']


In [31]:
# One-hot encode
df_encoded = pd.get_dummies(df, columns=ohe_cols, drop_first=False)


6. TEMPORAL FEATURES

In [33]:
df_encoded['course_start_date'] = pd.to_datetime(df_encoded['course_start_date'], errors='coerce')
df_encoded['assessment_date'] = pd.to_datetime(df_encoded['assessment_date'], errors='coerce')

In [34]:
# Week number relative to course start
df_encoded['week_number'] = (
    (df_encoded['assessment_date'] - df_encoded['course_start_date']).dt.days // 7
) + 1

In [36]:
# Days since course start
df_encoded['days_since_start'] = (
    df_encoded['assessment_date'] - df_encoded['course_start_date']
).dt.days

7. FEATURE ENGINEERING (multi-source)

In [37]:
# Engagement score (LMS clicks + time + forum activity)
df_encoded['engagement_score'] = (
    df_encoded['click_count'].fillna(0) +
    df_encoded['time_spent_lms'].fillna(0) +
    df_encoded['forum_chat_activity'].fillna(0)
)

In [38]:
# Assessment performance ratio
df_encoded['assessment_ratio'] = (
    df_encoded['assessment_score'] / df_encoded['assessment_max_score']
)

In [39]:
# Weighted CA contribution
df_encoded['weighted_score'] = df_encoded['assessment_ratio'] * df_encoded['ca_weight']

In [40]:
# Low attendance flag
df_encoded['low_attendance_flag'] = (df_encoded['attendance_percentage'] < 75).astype(int)

8. SCALE NUMERICAL FEATURES

In [41]:
# Recompute numerical columns after feature engineering
final_numerical_cols = df_encoded.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [42]:
# Exclude IDs from scaling
exclude_from_scaling = ['student_id', 'week_number']
scale_cols = [c for c in final_numerical_cols if c not in exclude_from_scaling]

In [43]:
scaler = StandardScaler()
df_encoded[scale_cols] = scaler.fit_transform(df_encoded[scale_cols])

9. FINAL CHECK

In [44]:
print(df_encoded.shape)
df_encoded.head()

(6124, 8745)


,gpa,module_credits,ca_weight,final_exam_weight,ca_marks,final_exam_marks,total_marks,lecture_attendance,attendance_percentage,assignment_marks,...,module_id_MOD-IS2240,module_id_MOD-IS2901,pass_fail_status_Fail,pass_fail_status_Pass,week_number,days_since_start,engagement_score,assessment_ratio,weighted_score,low_attendance_flag
0,0.500158,0.318116,-0.084567,0.084567,0.197593,0.378915,0.615265,1.411534,1.555238,3.522833,...,False,False,False,True,6,-0.245939,0.779902,0.565133,0.109386,-1.273411
1,0.285854,2.303993,2.511376,-2.511376,2.583137,-2.047697,0.299880,-0.535445,-0.597368,1.860477,...,False,False,False,True,8,0.893775,0.039775,0.274409,2.536483,0.785292
2,0.285854,-0.674822,-0.517225,0.517225,-0.524921,0.295045,-0.194842,0.172547,0.109537,-0.602831,...,False,False,False,True,8,0.893775,0.039775,0.274409,-0.407273,0.785292
3,0.285854,0.318116,-0.084567,0.084567,-0.325607,-0.392681,-0.757588,-0.712443,-0.756821,-0.361034,...,False,False,False,True,8,0.893775,0.039775,0.274409,0.013263,0.785292
4,0.285854,-0.674822,-0.517225,0.517225,-0.686864,-0.135482,-0.831797,-0.535445,-0.485752,-0.799291,...,False,False,False,True,8,0.893775,0.039775,0.274409,-0.407273,0.785292
